# Test metering

To test metering you put records into your DynamoDB metering table.
The metering records must match dimensions from your product definition
and you provide a customer AWS Account Id from a customer that 
purchased your product.

In [17]:
import boto3
import urllib.parse as urlparse 
import json
import random
import shutil
import time

from datetime import datetime, timezone

## Settings
Replace `stack_name` and `customer_identifier` with the values from
your environment.

Values such as productId, DynamoDB table and Lambda hourly function will
be retrieved from your CloudFormation stack.

In [3]:
# replace the value for stack_name with your value
#stack_name = 'REPLACE_WITH_YOUR_STACKNAME'
stack_name = 'eb-saas-sub'

# use the license arn (preferred) or customers AWS Account Id as customer_identifier
#customer_identifier = 'REPLACE_WITH_LICENSE_ARN' # license arn
customer_identifier = 'arn:aws:license-manager::294406891311:license:l-69040b4a916e453490dfc4d6ce097b10' # customers AWS Account id

In [4]:
PROFILE = 'default' # AWS profile to use
REGION = 'us-east-1' # AWS region

SESSION = boto3.Session(profile_name=PROFILE, region_name=REGION)

## Functions

Some functions to get product definition, put metering records in DynamoDB and scan a DynamoDB table.

In [5]:
def get_aws_account_id_for_license_arn(table_name, customer_identifier):
    dynamodb = SESSION.resource('dynamodb')
    table = dynamodb.Table(table_name)

    response = table.get_item(Key={'customerIdentifier': customer_identifier})
    item = response.get('Item')

    #print(json.dumps(item, indent=2))

    return item.get('customerAwsAccountId', 'NONE')


def get_marketplace_product(product_id):
    client = SESSION.client('marketplace-catalog')

    response = client.describe_entity(
        Catalog='AWSMarketplace',
        EntityId=product_id
    )
    
    return response

def create_metering_item(customer_identifier, customer_aws_account_id, dimension_name, dimension_value):
    #create_timestamp = f"{int(time.time())}" # Seconds precision, risk to overwrite records
    create_timestamp = f"{int(time.time_ns())}" # Nanosecond precision, virtually unique
    item = {
        "create_timestamp": {
            "N": create_timestamp
        },
        "customerIdentifier": {
            "S": customer_identifier
        },
        "customerAwsAccountId": {
            "S": customer_aws_account_id
        },
        "dimension_usage": {
            "L": [
            {
                "M": {
                "dimension": {
                    "S": dimension_name
                },
                "value": {
                    "N": f'{dimension_value}'
                }
                }
            }
            ]
        },
        "metering_pending": {
            "S": "true"
        }
    }
    
    return item


def scan_table(table_name):
    dynamodb = boto3.resource('dynamodb')
    table = dynamodb.Table(table_name)
    
    response = table.scan()
    return response['Items']


## Get stack values

Get your product id, DynamoDB metering table names and Lambda hourly funtion name from your CloudFormation stack.

In [6]:
product_id = None
metering_table_name = None
subscribers_table_name = None
lambda_hourly_function_name = None

cfn = SESSION.client('cloudformation')

# get product_id from stack parameters
response = cfn.describe_stacks(StackName=stack_name)
parameters = response['Stacks'][0]['Parameters']
for parameter in parameters:
  if parameter['ParameterKey'] == 'ProductId':
    product_id = parameter['ParameterValue']
    break

# get other values from stack resources
paginator = cfn.get_paginator('list_stack_resources')

for page in paginator.paginate(StackName=stack_name):
    for resource in page['StackResourceSummaries']:
        #print(json.dumps(resource, indent=2, default=str))
        #print(f"{resource['ResourceType']}: {resource['LogicalResourceId']}")
        #if resource['LogicalResourceId'] == 'AWSMarketplaceSubscribers':
        #    print(f"Subscribers table name: {resource['PhysicalResourceId']}")
        if resource['LogicalResourceId'] == 'AWSMarketplaceMeteringRecords':
            #print(f"Metering table name: {resource['PhysicalResourceId']}")
            metering_table_name = resource['PhysicalResourceId']
        if resource['LogicalResourceId'] == 'AWSMarketplaceSubscribers':
            #print(f"Metering table name: {resource['PhysicalResourceId']}")
            subscribers_table_name = resource['PhysicalResourceId']
        if resource['LogicalResourceId'] == 'Hourly':
            #print(f"Hourly Lambda function name: {resource['PhysicalResourceId']}")
            lambda_hourly_function_name = resource['PhysicalResourceId']


print(f"{'stack_name:':<35}{stack_name}")
print("-" * 60)
print(f"{'product_id:':<35}{product_id}")
print(f"{'metering_table_name:':<35}{metering_table_name}")
print(f"{'subscribers_table_name:':<35}{subscribers_table_name}")
print(f"{'lambda_hourly_function_name:':<35}{lambda_hourly_function_name}")
print(f"{'customer_identifier:':<35}{customer_identifier}")

stack_name:                        eb-saas-sub
------------------------------------------------------------
product_id:                        prod-bvjpvb6pzqiwk
metering_table_name:               MPMeteringSub
subscribers_table_name:            MPSubscribersSub
lambda_hourly_function_name:       eb-saas-sub-Hourly-RLscQ7F67CHA
customer_identifier:               arn:aws:license-manager::294406891311:license:l-69040b4a916e453490dfc4d6ce097b10


## Create metering entries

To create metering entries you get the usage dimensions from your product.
Then you create metering dimensions list `usage_dimensions` with randomized units.

Then use `create_metering_item` to put metering entries into the DynamoDB metering table.

## Get AWS account id
When you use the license arn (preferred) as customer identifier, retrieve the related
AWS account id from DynamoDB subscribers table.

In [27]:
customer_aws_account_id = customer_identifier

if customer_identifier.startswith('arn:aws:license-manager:'):
  print('customer identifier is a license arn, getting account id from DynamoDB')
  customer_aws_account_id = get_aws_account_id_for_license_arn(subscribers_table_name, customer_identifier)

print(f"customer_identifier: {customer_identifier} customer_aws_account_id: {customer_aws_account_id}")

customer identifier is a license arn, getting account id from DynamoDB
customer_identifier: arn:aws:license-manager::294406891311:license:l-69040b4a916e453490dfc4d6ce097b10 customer_aws_account_id: 017032326569


## Create dimensions

Create usage dimensions from product definition. Units are created randomly.

In [29]:
usage_dimensions = []
unit_min = 1
unit_max = 10

for dimension in product_def['DetailsDocument']['Dimensions']:
  if dimension['Types'][0] in 'ExternallyMetered':
    #print('payg')
    #print(dimension['Key'])
    usage_dimensions.append({dimension['Key']: random.randint(unit_min, unit_max)})
  print(f"dimension: {dimension}")

print("\nUsage dimensions (units randomized):")
print(usage_dimensions)

dimension: {'Name': 'Usage 1', 'Description': 'Usage 1', 'Key': 'usage_1', 'Unit': 'Units', 'Types': ['ExternallyMetered']}
dimension: {'Name': 'Usage 2', 'Description': 'Usage 2', 'Key': 'usage_2', 'Unit': 'Units', 'Types': ['ExternallyMetered']}
dimension: {'Name': 'Usage 3', 'Description': 'Usage 3', 'Key': 'usage_3', 'Unit': 'Units', 'Types': ['ExternallyMetered']}

Usage dimensions (units randomized):
[{'usage_1': 8}, {'usage_2': 2}, {'usage_3': 2}]


## Store metering entries in DynamoDB

Iterate of the `usage_dimensions` list and put metering entries into the metering table.

In [ ]:
ddb = SESSION.client('dynamodb')
for dimension in usage_dimensions:
    for dimension_name, dimension_value in dimension.items():
        print(f"metering dimension: {dimension_name} value: {dimension_value}")
        item = create_metering_item(customer_identifier, customer_aws_account_id, dimension_name, dimension_value)
        print(f"metering item:\n{json.dumps(item, indent=2, default=str)}")

        response = ddb.put_item(
            TableName=metering_table_name,
            Item=item
        )
        print(json.dumps(response, indent=2, default=str))
        print('-' * 50)
        # time.sleep(2) # not required for nanoseconds precision

metering dimension: usage_1 value: 1
metering item:
{
  "create_timestamp": {
    "N": "1772642524817792341"
  },
  "customerIdentifier": {
    "S": "arn:aws:license-manager::294406891311:license:l-69040b4a916e453490dfc4d6ce097b10"
  },
  "customerAwsAccountId": {
    "S": "017032326569"
  },
  "dimension_usage": {
    "L": [
      {
        "M": {
          "dimension": {
            "S": "usage_1"
          },
          "value": {
            "N": "1"
          }
        }
      }
    ]
  },
  "metering_pending": {
    "S": "true"
  }
}
{
  "ResponseMetadata": {
    "RequestId": "UB494M69NT0O3C8J3487JPJO7FVV4KQNSO5AEMVJF66Q9ASUAAJG",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "server": "Server",
      "date": "Wed, 04 Mar 2026 16:42:04 GMT",
      "content-type": "application/x-amz-json-1.0",
      "content-length": "2",
      "connection": "keep-alive",
      "x-amzn-requestid": "UB494M69NT0O3C8J3487JPJO7FVV4KQNSO5AEMVJF66Q9ASUAAJG",
      "x-amz-crc32": "2745614147"
    

## Get entries from the metering table

Scan the metering table.

Unprocessed item look similar to:

```
{
  "dimension_usage": [
    {
      "dimension": "usage_2",
      "value": "3"
    }
  ],
  "metering_pending": "true",
  "create_timestamp": "1763396941",
  "customerIdentifier": "944681004585"
}
```

Processed records have a `metering_failed` boolean key and a `metering_response` key for example:

```
"metering_failed": false,
  "dimension_usage": [
    {
      "dimension": "usage_1",
      "value": "3"
    }
  ],
  "create_timestamp": "1763392067",
  "customerIdentifier": "944681004585",
  "metering_response": "{\"$metadata\":{\"httpStatusCode\":200,\"requestId\":\"828fd9e5-ee21-4b0f-9656-82a86b4e49c2\",\"attempts\":1,\"totalRetryDelay\":0},\"Results\":[{\"MeteringRecordId\":\"eb0e9db9-c90e-45fa-84c4-6239a68fb360\",\"Status\":\"Success\",\"UsageRecord\":{\"CustomerAWSAccountId\":\"944681004585\",\"Dimension\":\"usage_1\",\"Quantity\":3,\"Timestamp\":\"2025-11-17T16:01:34.018Z\"}},{\"MeteringRecordId\":\"6756c73c-21fd-4821-a5d5-57a6d891f103\",\"Status\":\"Success\",\"UsageRecord\":{\"CustomerAWSAccountId\":\"944681004585\",\"Dimension\":\"usage_2\",\"Quantity\":6,\"Timestamp\":\"2025-11-17T16:01:34.018Z\"}},{\"MeteringRecordId\":\"c65d412f-aedf-4e89-afa3-a1d238277b63\",\"Status\":\"Success\",\"UsageRecord\":{\"CustomerAWSAccountId\":\"944681004585\",\"Dimension\":\"usage_3\",\"Quantity\":3,\"Timestamp\":\"2025-11-17T16:01:34.018Z\"}}],\"UnprocessedRecords\":[]}"
}
```

In [30]:
# Usage
items = scan_table(metering_table_name)
for item in items:
    print(json.dumps(item, indent=2, default=str))
    print('-' * 50)

print(f"\nscaned metering table: {metering_table_name}")

{
  "metering_failed": false,
  "dimension_usage": [
    {
      "dimension": "usage_1",
      "value": "1"
    }
  ],
  "customerAwsAccountId": "017032326569",
  "create_timestamp": "1772641125876728633",
  "customerIdentifier": "arn:aws:license-manager::294406891311:license:l-69040b4a916e453490dfc4d6ce097b10",
  "metering_response": "{\"Results\":[{\"UsageRecord\":{\"Timestamp\":\"2026-03-04T18:13:43.161Z\",\"Dimension\":\"usage_1\",\"Quantity\":2,\"CustomerAWSAccountId\":\"017032326569\",\"LicenseArn\":\"arn:aws:license-manager::294406891311:license:l-69040b4a916e453490dfc4d6ce097b10\"},\"MeteringRecordId\":\"0c8e93dc-bcd4-40ab-9039-f78b8d11d967\",\"Status\":\"Success\"},{\"UsageRecord\":{\"Timestamp\":\"2026-03-04T18:13:43.161Z\",\"Dimension\":\"usage_2\",\"Quantity\":4,\"CustomerAWSAccountId\":\"017032326569\",\"LicenseArn\":\"arn:aws:license-manager::294406891311:license:l-69040b4a916e453490dfc4d6ce097b10\"},\"MeteringRecordId\":\"e8c45efc-7dc0-4a23-a792-23ebe274e17a\",\"Status\"

## Trigger metering hourly function

The Lambda function that meters hourly is automatically triggered by an
Amazon EventBridge rule. For testing purposes you can also
invoke the funtion manually.

In [39]:
lmbd = SESSION.client('lambda')

response = lmbd.invoke(
    FunctionName=lambda_hourly_function_name,
    Payload=json.dumps({'start': 'metering'})
)

print(f"response:\n{json.dumps(response, indent=2, default=str)}")
print(f"Payload:\n{json.loads(response['Payload'].read())}")


response:
{
  "ResponseMetadata": {
    "RequestId": "ac78a10e-509c-4564-b65c-0f71f4616d47",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Wed, 04 Mar 2026 18:18:03 GMT",
      "content-type": "application/json",
      "content-length": "4",
      "connection": "keep-alive",
      "x-amzn-requestid": "ac78a10e-509c-4564-b65c-0f71f4616d47",
      "x-amzn-remapped-content-length": "0",
      "x-amz-executed-version": "$LATEST",
      "x-amzn-trace-id": "Root=1-69a87759-43d1393f3f22af0b71b1bf1d;Parent=1576741b78b9ca22;Sampled=0;Lineage=1:46392bce:0"
    },
    "RetryAttempts": 0
  },
  "StatusCode": 200,
  "ExecutedVersion": "$LATEST",
  "Payload": "<botocore.response.StreamingBody object at 0x7cad0b4daaa0>"
}
Payload:
True
